In [1]:
!pip install unsloth
!pip install -U --no-deps xformers trl peft accelerate bitsandbytes

# 2. Import and check GPU
import torch
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print(" ERROR: No GPU detected. Please change Runtime Type to T4 GPU.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/69.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/18

In [2]:
from unsloth import FastLanguageModel
import torch

# Load the model (DeepSeek-R1-Distill-Qwen-1.5B)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = 8192,
    load_in_4bit = True,
    device_map = "auto", # Ensure it maps to the T4 correctly
)

# Enable 2x faster inference kernels
FastLanguageModel.for_inference(model)

# Set padding side to left for batching consistency
tokenizer.padding_side = "left"

# R1-Specific Padding Fix
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [3]:
import json
from collections import Counter
import os

# Update to match your new FINAL file name
INPUT_FILE = "test_suite_200_FINAL.jsonl"

difficulty_counts = Counter()

if not os.path.exists(INPUT_FILE):
    print(f" Error: {INPUT_FILE} not found. Please upload it to Colab!")
else:
    with open(INPUT_FILE, "r") as f:
        for line in f:
            try:
                task = json.loads(line)
                # Normalize to lowercase to avoid "Easy" vs "easy" mismatches
                diff = str(task.get('difficulty', 'unknown')).lower()
                difficulty_counts[diff] += 1
            except json.JSONDecodeError:
                continue

    print("--- Test Suite Difficulty Distribution ---")
    for diff, count in sorted(difficulty_counts.items()):
        print(f"{diff:12}: {count} problems")
    print(f"{'-'*42}")
    print(f"Total       : {sum(difficulty_counts.values())} problems")

--- Test Suite Difficulty Distribution ---
easy        : 60 problems
hard        : 60 problems
medium      : 80 problems
------------------------------------------
Total       : 200 problems


In [7]:
import json
import os
import torch
import time
from tqdm.auto import tqdm
from unsloth import FastLanguageModel

INPUT_FILE = "test_suite_200_FINAL.jsonl" # Updated to your new clean file
OUTPUT_FILE = "model_solutions_batched.jsonl"
BATCH_SIZE = 10  # Optimized for Tesla T4 (16GB)

# Limits based on difficulty to prevent OOM while allowing complex reasoning
LIMITS = {"easy": 2048, "medium": 4096, "hard": 8192, "very_hard": 8192}

solved_ids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r") as f:
        for line in f:
            try:
                solved_ids.add(json.loads(line)["id"])
            except: continue

with open(INPUT_FILE, "r") as f:
    all_tasks = [json.loads(line) for line in f]

# Filter out what's already done
tasks_to_run = [t for t in all_tasks if t["id"] not in solved_ids]
print(f"Total: {len(all_tasks)} | Already Solved: {len(solved_ids)} | Remaining: {len(tasks_to_run)}")

with open(OUTPUT_FILE, "a") as out_f:
    pbar = tqdm(range(0, len(tasks_to_run), BATCH_SIZE), desc="Batched Solving")

    for i in pbar:
        batch = tasks_to_run[i : i + BATCH_SIZE]

        # Determine max tokens for this specific batch based on the hardest problem present
        current_batch_limit = max([LIMITS.get(t['difficulty'].lower(), 4096) for t in batch])

        # Strengthened System Prompt to force complete code generation
        prompts = [
            f"System: You are a CP expert. Provide a complete, runnable Python solution. "
            f"Think step-by-step inside <think> and provide final implementation inside <code>. "
            f"Do not use placeholders like 'pass' or 'TODO'.\nUser: {t['problem']}"
            for t in batch
        ]

        inputs = tokenizer(prompts, padding=True, return_tensors="pt").to("cuda")

        start_time = time.time()
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens = current_batch_limit,
                use_cache = True,
                temperature = 0.6,
                repetition_penalty = 1.2, # Slightly increased to prevent looping
                stop_strings = ["</code>"],
                tokenizer = tokenizer,
            )
        end_time = time.time()

        # Post-Processing & Metrics
        total_tokens_in_batch = 0
        for idx, output in enumerate(outputs):
            # Calculate tokens generated (excluding the prompt)
            prompt_len = inputs['input_ids'][idx].shape[0]
            tokens_gen = output.shape[0] - prompt_len
            total_tokens_in_batch += tokens_gen

            # Save results immediately
            out_f.write(json.dumps({
                "id": batch[idx]["id"],
                "difficulty": batch[idx]["difficulty"],
                "model_output": tokenizer.decode(output, skip_special_tokens=True),
                "tokens_used": tokens_gen,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            }) + "\n")

        # UI Updates
        vram_used = torch.cuda.memory_reserved() / 1024**3
        tps = total_tokens_in_batch / (end_time - start_time)
        pbar.set_postfix({"tk/s": f"{tps:.1f}", "VRAM": f"{vram_used:.1f}GB"})
        out_f.flush() # Ensure data is written even if Colab crashes

print(f"\n All problems processed! Results saved to {OUTPUT_FILE}")

Total: 200 | Already Solved: 0 | Remaining: 200


Batched Solving:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 